# 04 - Continuous OptBinning for Amount + Explained Variance

This notebook applies continuous optimal binning against deal amount and reports:
- binning-based IV-like score from continuous tables
- explained variance from bin-mean predictions
- ANOVA F-statistic over bin groups

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import f_oneway
from sklearn.metrics import explained_variance_score
from optbinning import ContinuousOptimalBinning

pd.set_option('display.max_columns', 200)

In [2]:
df_load = pd.read_parquet('../../data/intermediate/df_train_stratified.parquet')
print('shape:', df_load.shape)
df_load.head(2)

shape: (54617, 37)


,Opportunity Number,Supplies Group,Supplies Subgroup,Region,Route To Market,Elapsed Days In Sales Stage,Opportunity Result,Sales Stage Change Count,Total Days Identified Through Closing,Total Days Identified Through Qualified,Opportunity Amount USD,Client Size By Revenue (USD),Client Size By Employee Count,Revenue From Client Past Two Years (USD),Competitor Type,Ratio Days Identified To Total Days,Ratio Days Validated To Total Days,Ratio Days Qualified To Total Days,Deal Size Category (USD),total_days_zero,Opportunity Result Bool,opportunity_amount_weirdness,row_position,flag_0_days,flag_ratio_problem,flag_zero_opportunity_amount,flag_outlier_opportunity_amount,flag_outlier_total_days,flag_totally_repeated_row,flag_partially_repeated_row,partial_repeat_is_latest_id_appearance,flag_only_identified,flag_weirdness_over_75th_pct,problem_tags,problem_count,amount_qbin,stratify_key
0,7062187,Car Accessories,Towing & Hitches,Northeast,Fields Sales,41,Loss,4,41,41,200000,100K or less,1K or less,"25,000 - 50,000",NaN,0.84058,0.15942,0.0,40K to 50K,False,False,19.931465,55880,False,False,False,False,False,False,False,False,False,True,[weirdness_over_75th_pct],1,"(131000.0, 220000.0]","0__(131000.0, 220000.0]"
1,9718029,Car Accessories,Garage & Car Care,Midwest,Telesales,7,Loss,2,6,6,28763,100K or less,1K or less,0 (No business),Unknown,1.00000,0.00000,0.0,20K to 30K,False,False,4.578548,43421,False,False,False,False,False,False,False,False,True,False,[flag_only_identified],1,"(20000.0, 30000.0]","0__(20000.0, 30000.0]"


In [3]:
columns_to_use = ['Supplies Group', 'Supplies Subgroup', 'Region',
       'Route To Market', 'Elapsed Days In Sales Stage', 'Opportunity Result',
       'Sales Stage Change Count', 'Total Days Identified Through Closing',
       'Total Days Identified Through Qualified', 'Opportunity Amount USD',
       'Client Size By Revenue (USD)', 'Client Size By Employee Count',
       'Revenue From Client Past Two Years (USD)', 'Competitor Type',
       'Ratio Days Identified To Total Days',
       'Ratio Days Validated To Total Days',
       'Ratio Days Qualified To Total Days', 'Deal Size Category (USD)', "Opportunity Result Bool"]

df = df_load[columns_to_use].copy()

In [4]:
df['target_amount'] = df['Opportunity Amount USD'].astype(float)
df['target_log_amount'] = np.log1p(df['target_amount'])
print('shape:', df.shape)
print(df[['target_amount', 'target_log_amount']].describe())

shape: (54617, 21)
        target_amount  target_log_amount
count    54617.000000       54617.000000
mean     91610.549225          10.314420
std     133074.048073           2.292708
min          0.000000           0.000000
25%      15000.000000           9.615872
50%      49000.000000          10.799596
75%     105000.000000          11.561725
max    1000000.000000          13.815512


## Fit continuous optimal binning and compute explained variance

In [3]:
excluded = {'Opportunity Amount USD', 'target_amount', 'target_log_amount', 'Opportunity Number'}
features = [c for c in df.columns if c not in excluded]

rows = []
failed = []
y = df['target_log_amount']
for col in features:
    x = df[col]
    dtype = 'numerical' if pd.api.types.is_numeric_dtype(x) else 'categorical'

    try:
        cob = ContinuousOptimalBinning(name=col, dtype=dtype)
        cob.fit(x, y)

        bt = cob.binning_table.build()
        iv_total = float(bt.iloc[-1]['IV']) if 'IV' in bt.columns else np.nan

        idx = cob.transform(x, metric='indices')
        work = pd.DataFrame({'idx': idx, 'y': y})
        bin_mean = work.groupby('idx', dropna=False)['y'].mean()
        yhat = work['idx'].map(bin_mean).astype(float).to_numpy()
        ev = float(explained_variance_score(y, yhat))

        groups = [g['y'].values for _, g in work.groupby('idx') if len(g) > 1]
        if len(groups) >= 2:
            f_stat = float(f_oneway(*groups).statistic)
        else:
            f_stat = np.nan

        rows.append({
            'variable': col,
            'dtype': dtype,
            'status': cob.status,
            'iv_continuous': iv_total,
            'explained_variance': ev,
            'anova_f': f_stat
        })
    except Exception as e:
        failed.append({'variable': col, 'error': str(e)[:200]})

amount_bin_df = pd.DataFrame(rows)
if not amount_bin_df.empty:
    amount_bin_df = amount_bin_df.sort_values('explained_variance', ascending=False).reset_index(drop=True)
failed_df = pd.DataFrame(failed)
print('processed:', len(amount_bin_df), 'failed:', len(failed_df))
amount_bin_df.head(20)

processed: 17 failed: 0


,variable,dtype,status,iv_continuous,explained_variance,anova_f
0,Deal Size Category (USD),categorical,OPTIMAL,1.444898,0.673988,32258.820970
1,Total Days Identified Through Qualified,numerical,OPTIMAL,0.266347,0.021850,174.271322
2,Total Days Identified Through Closing,numerical,OPTIMAL,0.262242,0.020670,164.655558
3,Route To Market,categorical,OPTIMAL,0.321507,0.019698,1567.772906
4,Supplies Subgroup,categorical,OPTIMAL,0.250268,0.018890,214.586543
5,Competitor Type,categorical,OPTIMAL,0.190901,0.013290,1050.891894
6,Opportunity Result,categorical,OPTIMAL,0.205392,0.011483,906.368208
7,Ratio Days Identified To Total Days,numerical,OPTIMAL,0.205773,0.010200,200.996800
8,Ratio Days Validated To Total Days,numerical,OPTIMAL,0.203219,0.009202,120.768005
9,Ratio Days Qualified To Total Days,numerical,OPTIMAL,0.127896,0.005017,98.344976


In [4]:
amount_bin_df.to_csv('../data/processed/amount_optbinning_explained_variance.csv', index=False)
if not failed_df.empty:
    failed_df.to_csv('../data/processed/amount_optbinning_failed.csv', index=False)
print('saved: ../data/processed/amount_optbinning_explained_variance.csv')

saved: ../data/processed/amount_optbinning_explained_variance.csv


## Top variable continuous binning table

In [5]:
if amount_bin_df.empty:
    print('No successful continuous optbinning fits to display.')
else:
    top_var = amount_bin_df.loc[0, 'variable']
    dtype = 'numerical' if pd.api.types.is_numeric_dtype(df[top_var]) else 'categorical'
    cob = ContinuousOptimalBinning(name=top_var, dtype=dtype)
    cob.fit(df[top_var], df['target_log_amount'])
    print('Top variable:', top_var)
    display(cob.binning_table.build())

Top variable: Deal Size Category (USD)


,Bin,Count,Count (%),Sum,Std,Mean,Min,Max,Zeros count,WoE,IV
0,[10K or less],12095,0.155014,78898.973493,3.263002,6.523272,0.000000,9.210340,2047,-3.791020,0.587663
1,[10K to 20K],15123,0.193822,145466.360740,0.292173,9.618883,9.210440,10.126631,0,-0.695410,0.134786
2,[20K to 30K],11968,0.153387,124528.893866,0.197771,10.405155,10.126671,10.819778,0,0.090863,0.013937
3,[30K to 40K],13628,0.174662,150423.506464,0.215005,11.037827,10.819798,11.512925,0,0.723535,0.126374
4,[40K to 50K],18074,0.231644,213454.567664,0.273613,11.810035,11.512935,12.429216,0,1.495743,0.346479
5,"[50K to 60K, More than 60K]",7137,0.091471,92000.347383,0.387359,12.890619,12.429220,13.815512,0,2.576327,0.235658
6,Special,0,0.000000,0.000000,NaN,0.000000,NaN,NaN,0,-10.314292,0.000000
7,Missing,0,0.000000,0.000000,NaN,0.000000,NaN,NaN,0,-10.314292,0.000000
Totals,,78025,1.000000,804772.649609,,10.314292,0.000000,13.815512,2047,30.001481,1.444898
